# Imports

In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
from rdkit import Chem

PROJECT_ROOT = Path.cwd().parent  # si el notebook esta en /notebooks
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths_and_constants import Paths
_P = Paths()

# Functions

def is_ion_smiles(smiles: str) -> bool:
    """
    Returns true if SMILES corresponds to a monoatomic ion.
    """
    if smiles is None or (isinstance(smiles, float) and pd.isna(smiles)):
        return False

    from rdkit import RDLogger
    RDLogger.DisableLog('rdApp.error')

    m = Chem.MolFromSmiles(str(smiles))
    
    RDLogger.EnableLog('rdApp.error')
    
    if m is None:
        return False

    # Ion = un solo átomo, sin enlaces
    if m.GetNumAtoms() != 1:
        return False

    atom = m.GetAtomWithIdx(0)
    return atom.GetSymbol().upper() in ion_symbols

def add_negatives(
    df,
    seq_col="protein_sequence",
    lig_col="smiles",
    neg_per_pos=1,
    seed=42,
    max_tries=1000,
):
    rng = np.random.default_rng(seed)

    df = df.copy()
    if "label" not in df.columns:
        df["label"] = 1

    positives = set(zip(df[seq_col], df[lig_col]))
    ligands_filtered = set(df[lig_col].dropna().unique())

    print(f"Initial Dataset Shape: {df.shape}")
    print(f"Total Unique Ligands Available: {len(ligands_filtered)}")
    print(f"Total Unique Proteins to Process: {df[seq_col].nunique()}")

    negatives = set()

    for p, grp in df.groupby(seq_col):
        pos_ligands_p = set(grp[lig_col].dropna().unique())
        if not pos_ligands_p:
            continue

        candidate_ligands = list(ligands_filtered - pos_ligands_p)
        n_neg = neg_per_pos * len(pos_ligands_p)
        if not candidate_ligands:
            print(f"Skipping {p[:10]}... No candidate ligands available.")
            continue

        if len(candidate_ligands) >= n_neg:
            chosen = rng.choice(candidate_ligands, size=n_neg, replace=False)
        else:
            chosen = rng.choice(candidate_ligands, size=n_neg, replace=True)

        for l_neg in chosen:
            if (p, l_neg) not in positives:
                negatives.add((p, l_neg))

        if len(candidate_ligands) < n_neg and max_tries > 0:
            tries = 0
            while len(negatives) < n_neg and tries < max_tries:
                l_neg = rng.choice(candidate_ligands)
                if (p, l_neg) not in positives:
                    negatives.add((p, l_neg))
                tries += 1

    neg_df = pd.DataFrame(list(negatives), columns=[seq_col, lig_col])
    neg_df["label"] = 0

    # columns alignment
    for col in df.columns:
        if col not in neg_df.columns:
            neg_df[col] = np.nan
    neg_df = neg_df[df.columns]

    out_df = pd.concat([df, neg_df], ignore_index=True)
    return out_df

# Dataprep for Biolip

In [2]:
biol = pd.read_csv(_P.data_interim_dir / "biolip" / "BioLiP_nr.csv", low_memory=False)
print(str(biol.shape) + " before filtering")
print(biol.columns)
biol = biol[(biol['ec_number'].notna())]
print(str(biol.shape) + " after filtering for EC number")
biol = biol[(biol['uniprot_id'].notna())]
print(str(biol.shape) + " after filtering for UniProt ID")
biol = biol[(biol['ligand_id'].notna())]
print(str(biol.shape) + " after filtering for Ligand ID")

(86458, 21) before filtering
Index(['pdb_id', 'chain_id', 'resolution', 'binding_site_id', 'ligand_id',
       'ligand_chain_id', 'ligand_serial_number', 'binding_site_residues',
       'nearby_residues', 'ligand_contact_residues', 'catalytic_site_residues',
       'ec_number', 'go_terms', 'binding_affinity_1', 'binding_affinity_2',
       'binding_affinity_3', 'binding_affinity_4', 'uniprot_id', 'pubmed_id',
       'binding_site_range', 'protein_sequence'],
      dtype='str')
(68497, 21) after filtering for EC number
(68291, 21) after filtering for UniProt ID
(68206, 21) after filtering for Ligand ID


# BioLiP maps peptides to PEP, CCD maps PEP to phospholenolpyruvate, original code maps false interactions

ccd = pd.read_csv(
    _P.data_interim_dir / 'biolip' / "Components-smiles-cactvs.smi",
    sep="\t",
    header=None,
    names=["smiles", "ccd_id", "ligand_name"],
    dtype=str
)
ccd["ccd_id"] = ccd["ccd_id"].str.strip()
ccd["smiles"] = ccd["smiles"].str.strip()
ccd["ligand_name"] = ccd["ligand_name"].str.strip()

# Looks up (diccs) to map ccd_id -> (smiles, name)
ccd_exact = ccd.drop_duplicates("ccd_id").set_index("ccd_id")[["smiles", "ligand_name"]]
ccd_upper = (
    ccd.assign(ccd_id_upper=ccd["ccd_id"].str.upper())
       .drop_duplicates("ccd_id_upper")
       .set_index("ccd_id_upper")[["smiles", "ligand_name"]]
)

# Prepare BioLiP IDs
biol2 = biol.copy()
biol2["ligand_id_raw"] = biol2["ligand_id"].astype(str).str.strip()

# Map (case-sensitive)
biol2["smiles"] = biol2["ligand_id_raw"].map(ccd_exact["smiles"])
biol2["ligand_name"] = biol2["ligand_id_raw"].map(ccd_exact["ligand_name"])
biol2["map_status"] = "exact"

# Fallback to UPPER (only if not mapped yet and not polymer token)
polymer_tokens = {"dna", "rna", "pep", "peptide", "protein", "polypeptide"}

need = biol2["smiles"].isna()
ok = ~biol2["ligand_id_raw"].str.lower().isin(polymer_tokens)

idx = biol2.index[need & ok]
biol2.loc[idx, "smiles"] = biol2.loc[idx, "ligand_id_raw"].str.upper().map(ccd_upper["smiles"])
biol2.loc[idx, "ligand_name"] = biol2.loc[idx, "ligand_id_raw"].str.upper().map(ccd_upper["ligand_name"])
biol2.loc[idx, "map_status"] = "rescued_upper"

# Track whatever was blocked and not mapped
blocked = need & ~ok
biol2.loc[blocked, "map_status"] = "polymer_token_blocked"

still_unmapped = biol2["smiles"].isna() & ok
biol2.loc[still_unmapped, "map_status"] = "unmapped"
biol_smiles = biol2
print(biol_smiles.shape)
biol_smiles.dropna(subset=["smiles"], inplace=True)
print(biol_smiles.shape)
biol_smiles


In [3]:
# BioLiP maps peptides to PEP, CCD maps PEP to phospholenolpyruvate, original code maps false interactions

ccd = pd.read_csv(
    _P.data_interim_dir / 'biolip' / "Components-smiles-cactvs.smi",
    sep="\t",
    header=None,
    names=["smiles", "ccd_id", "ligand_name"],
    dtype=str
)
ccd["ccd_id"] = ccd["ccd_id"].str.strip()
ccd["smiles"] = ccd["smiles"].str.strip()
ccd["ligand_name"] = ccd["ligand_name"].str.strip()

# Looks up (diccs) to map ccd_id -> (smiles, name)
ccd_exact = ccd.drop_duplicates("ccd_id").set_index("ccd_id")[["smiles", "ligand_name"]]
ccd_upper = (
    ccd.assign(ccd_id_upper=ccd["ccd_id"].str.upper())
       .drop_duplicates("ccd_id_upper")
       .set_index("ccd_id_upper")[["smiles", "ligand_name"]]
)

# Prepare BioLiP IDs
biol2 = biol.copy()
biol2["ligand_id_raw"] = biol2["ligand_id"].astype(str).str.strip()

# Define polymers that should NOT be mapped as small molecules
polymer_tokens = {"dna", "rna", "pep", "peptide", "protein", "polypeptide"}
is_polymer = biol2["ligand_id_raw"].str.lower().isin(polymer_tokens)

# Initialize empty columns with object dtype to avoid LossySetitemError when inserting strings
biol2["smiles"] = pd.Series(np.nan, index=biol2.index, dtype=object)
biol2["ligand_name"] = pd.Series(np.nan, index=biol2.index, dtype=object)
biol2["map_status"] = "unmapped"

# Mark polymers so they bypass the mapping completely
biol2.loc[is_polymer, "map_status"] = "polymer_token_blocked"

# 1. Map (case-sensitive) ONLY for non-polymers
ok_to_map = ~is_polymer
exact_matches = biol2.loc[ok_to_map, "ligand_id_raw"].map(ccd_exact["smiles"]).notna()
idx_exact = biol2.index[ok_to_map & exact_matches]

biol2.loc[idx_exact, "smiles"] = biol2.loc[idx_exact, "ligand_id_raw"].map(ccd_exact["smiles"])
biol2.loc[idx_exact, "ligand_name"] = biol2.loc[idx_exact, "ligand_id_raw"].map(ccd_exact["ligand_name"])
biol2.loc[idx_exact, "map_status"] = "exact"

# 2. Fallback to UPPER (only if not mapped yet and not a polymer token)
need = biol2["smiles"].isna()
idx_upper = biol2.index[need & ok_to_map]

upper_matches = biol2.loc[idx_upper, "ligand_id_raw"].str.upper().map(ccd_upper["smiles"]).notna()
idx_upper_success = idx_upper[upper_matches]

biol2.loc[idx_upper_success, "smiles"] = biol2.loc[idx_upper_success, "ligand_id_raw"].str.upper().map(ccd_upper["smiles"])
biol2.loc[idx_upper_success, "ligand_name"] = biol2.loc[idx_upper_success, "ligand_id_raw"].str.upper().map(ccd_upper["ligand_name"])
biol2.loc[idx_upper_success, "map_status"] = "rescued_upper"

biol_smiles = biol2
print(biol_smiles.shape)
biol_smiles.dropna(subset=["smiles"], inplace=True)
print(biol_smiles.shape)
biol_smiles

(68206, 25)
(55269, 25)


,pdb_id,chain_id,resolution,binding_site_id,ligand_id,ligand_chain_id,ligand_serial_number,binding_site_residues,nearby_residues,ligand_contact_residues,...,binding_affinity_3,binding_affinity_4,uniprot_id,pubmed_id,binding_site_range,protein_sequence,ligand_id_raw,smiles,ligand_name,map_status
2,10mh,A,2.550,BS03,SAH,A,1,F18 A19 G20 L21 G23 E40 W41 D60 N304 S305,F18 A19 G20 L21 G23 E40 W41 D60 N304 S305,C81 E119 R163 R165,...,NaN,NaN,P05102,9925782.0,328,MIEIKDKQLTGLRFIDLFAGLGGFRLALESCGAECVYSNEWDKYAQ...,SAH,N[CH](CCSC[CH]1O[CH]([CH](O)[CH]1O)n2cnc3c(N)n...,s-adenosyl-l-homocysteine,exact
3,11as,A,2.500,BS01,ASN,A,1,D46 A74 Q116 Y218 S251 R255,D43 A71 Q113 Y215 S248 R252,D46 R100 Q116 D235 E248 S251,...,NaN,NaN,P00963,9437423.0,331,AYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLS...,ASN,N[CH](CC(N)=O)C(O)=O,asparagine,exact
4,11ba,A,2.060,BS01,UPA,A,1,K41 V43 N44 T45 N67 Q69 N71 A109 H119 F120,K41 V43 N44 T45 N67 Q69 N71 A109 H119 F120,H12 K41 H119 F120 D121,...,NaN,NaN,P00669,10082366.0,125,KESAAAKFERQHMDSGNSPSSSSNYCNLMMCCRKMTQGKCKPVNTF...,UPA,Nc1ncnc2n(cnc12)[CH]3O[CH](CO[P](O)(=O)O[CH]4[...,uridylyl-2'-5'-phospho-adenosine,exact
5,13pk,A,2.500,BS01,ADP,A,1,G217 A218 K223 L315 P340 G342 V343 E345 G375 D...,G213 A214 K219 L311 P336 G338 V339 E341 G371 D...,R39 K219 G376 G399,...,NaN,NaN,P07378,9000079.0,421,EKKSINECDLKGKKVLIRVDFNVPVKNGKITNDYRIRSALPTLKKV...,ADP,Nc1ncnc2n(cnc12)[CH]3O[CH](CO[P](O)(=O)O[P](O)...,adenosine-5'-diphosphate,exact
6,155c,A,2.500,BS01,HEM,A,1,C15 C18 H19 I46 A47 Y54 G55 I58 W70 Y78 V79 K9...,C15 C18 H19 I46 A47 Y54 G55 I58 W70 Y78 V79 K9...,NaN,...,NaN,NaN,P00096,180013.0,135,NEGDAAKGEKEFNKCKACHMIQAPDGTDIKGGKTGPNLYGVVGRKI...,HEM,CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=C...,protoporphyrin ix containing fe,exact
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82359,9ezj,C,2.612,BS01,ZIQ,C,1,R103 E105 W208 R211 T212 P213 F304,R64 E66 W148 R151 T152 P153 F244,NaN,...,NaN,NaN,P48775,39537789.0,501,LIYGNYLHLEKVLNAQELQSETKGNKIHDEHLFIITHQAYELWFKQ...,ZIQ,C[C](N)(Cc1c[nH]c2ccccc12)C(O)=O,alpha-methyl-l-tryptophan,exact
82360,9ezj,C,2.612,BS02,A1H8G,C,1,F72 H76 F129 F140 F153 Q154 S155 H328,F33 H37 F90 F101 F111 Q112 S113 H268,NaN,...,NaN,NaN,P48775,39537789.0,502,LIYGNYLHLEKVLNAQELQSETKGNKIHDEHLFIITHQAYELWFKQ...,A1H8G,CCOC(=O)C1=C(OC)N=C2[CH](CCCN2C1=O)N([CH](C)c3...,ethyl (9~{r})-2-methoxy-4-oxidanylidene-9-[[(1...,exact
82553,9fgp,A,1.490,BS01,ZN,A,1,E125 D164,E36 D75,NaN,...,NaN,NaN,Q6V702,NaN,301,SMDNFLTALAMREEDNRSGKLSSVIFIRDRNSHGQEISGYIDYAHR...,ZN,[Zn++],zinc ion,exact
86244,9pcy,A,-1.000,BS01,CU,A,1,H37 C84 H87,H37 C84 H87,H37 C84 H87 M92,...,NaN,NaN,P00287,1920431.0,100,LEVLLGSGDGSLVFVPSEFSVPSGEKIVFKNNAGFPHNVVFDEDEI...,CU,[Cu++],copper (ii) ion,exact


In [4]:
def is_ion_smiles(smiles: str) -> bool:
    """
    Returns true if SMILES corresponds to a monoatomic ion.
    """

    if smiles is None or (isinstance(smiles, float) and pd.isna(smiles)):
        return False

    from rdkit import RDLogger
    RDLogger.DisableLog('rdApp.error')
    
    m = Chem.MolFromSmiles(str(smiles))
    
    RDLogger.EnableLog('rdApp.error')
    
    if m is None:
        return False

    # Ion = un solo átomo, sin enlaces
    if m.GetNumAtoms() != 1:
        return False

    atom = m.GetAtomWithIdx(0)
    return atom.GetSymbol().upper() in ion_symbols

def add_negatives(
    df,
    seq_col="protein_sequence",
    lig_col="smiles",
    neg_per_pos=1,
    seed=42,
    max_tries=1000,
):
    rng = np.random.default_rng(seed)

    df = df.copy()
    if "label" not in df.columns:
        df["label"] = 1

    positives = set(zip(df[seq_col], df[lig_col]))
    ligands_filtered = set(df[lig_col].dropna().unique())

    print(f"Initial Dataset Shape: {df.shape}")
    print(f"Total Unique Ligands Available: {len(ligands_filtered)}")
    print(f"Total Unique Proteins to Process: {df[seq_col].nunique()}")

    negatives = set()

    for p, grp in df.groupby(seq_col):
        pos_ligands_p = set(grp[lig_col].dropna().unique())
        if not pos_ligands_p:
            continue

        candidate_ligands = list(ligands_filtered - pos_ligands_p)
        
        n_neg = min(neg_per_pos * len(pos_ligands_p), len(candidate_ligands))
        
        if not candidate_ligands:
            print(f"Skipping {p[:10]}... No candidate ligands available.")
            continue

        chosen = rng.choice(candidate_ligands, size=n_neg, replace=False)

        for l_neg in chosen:
            negatives.add((p, l_neg))

    neg_df = pd.DataFrame(list(negatives), columns=[seq_col, lig_col])
    neg_df["label"] = 0

    print(f"Generated {len(neg_df)} negative samples.")

    # columns alignment
    for col in df.columns:
        if col not in neg_df.columns:
            neg_df[col] = np.nan
    neg_df = neg_df[df.columns]

    out_df = pd.concat([df, neg_df], ignore_index=True)
    
    print("Final Output Label Distribution:")
    print(out_df["label"].value_counts())

    return out_df

In [5]:
ion_symbols = {
    "LI","NA","K","RB","CS",
    "MG","CA","SR","BA",
    "ZN","FE","CU","MN","CO","NI",
    "CD","HG","AL","CR","V","MO","W",
    "CL","BR","I","F",
    "AG","AU","PT","PB","SN"
}

# Ion mask
is_ion = biol_smiles["smiles"].apply(is_ion_smiles)

# final filter
biol_filtered = biol_smiles[~is_ion].copy()

print("original:", biol_smiles.shape)
print("removed ions:", int(is_ion.sum()))
print("kept:", biol_filtered.shape)

# sanity check
print("\nExamples removed as ions:")
print(
    biol_smiles.loc[is_ion, ["ligand_id_raw", "ligand_name", "smiles"]]
    .head(10)
)

[17:35:38] WARNING: not removing hydrogen atom without neighbors


original: (55269, 25)
removed ions: 18951
kept: (36318, 25)

Examples removed as ions:
   ligand_id_raw         ligand_name  smiles
21            CO     cobalt (ii) ion  [Co++]
22            CO     cobalt (ii) ion  [Co++]
23            MN  manganese (ii) ion  [Mn++]
24            MN  manganese (ii) ion  [Mn++]
25            CO     cobalt (ii) ion  [Co++]
26            CO     cobalt (ii) ion  [Co++]
30            CA         calcium ion  [Ca++]
33            CA         calcium ion  [Ca++]
50            MN  manganese (ii) ion  [Mn++]
57            CA         calcium ion  [Ca++]


In [6]:
biol_filtered

,pdb_id,chain_id,resolution,binding_site_id,ligand_id,ligand_chain_id,ligand_serial_number,binding_site_residues,nearby_residues,ligand_contact_residues,...,binding_affinity_3,binding_affinity_4,uniprot_id,pubmed_id,binding_site_range,protein_sequence,ligand_id_raw,smiles,ligand_name,map_status
2,10mh,A,2.550,BS03,SAH,A,1,F18 A19 G20 L21 G23 E40 W41 D60 N304 S305,F18 A19 G20 L21 G23 E40 W41 D60 N304 S305,C81 E119 R163 R165,...,NaN,NaN,P05102,9925782.0,328,MIEIKDKQLTGLRFIDLFAGLGGFRLALESCGAECVYSNEWDKYAQ...,SAH,N[CH](CCSC[CH]1O[CH]([CH](O)[CH]1O)n2cnc3c(N)n...,s-adenosyl-l-homocysteine,exact
3,11as,A,2.500,BS01,ASN,A,1,D46 A74 Q116 Y218 S251 R255,D43 A71 Q113 Y215 S248 R252,D46 R100 Q116 D235 E248 S251,...,NaN,NaN,P00963,9437423.0,331,AYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLS...,ASN,N[CH](CC(N)=O)C(O)=O,asparagine,exact
4,11ba,A,2.060,BS01,UPA,A,1,K41 V43 N44 T45 N67 Q69 N71 A109 H119 F120,K41 V43 N44 T45 N67 Q69 N71 A109 H119 F120,H12 K41 H119 F120 D121,...,NaN,NaN,P00669,10082366.0,125,KESAAAKFERQHMDSGNSPSSSSNYCNLMMCCRKMTQGKCKPVNTF...,UPA,Nc1ncnc2n(cnc12)[CH]3O[CH](CO[P](O)(=O)O[CH]4[...,uridylyl-2'-5'-phospho-adenosine,exact
5,13pk,A,2.500,BS01,ADP,A,1,G217 A218 K223 L315 P340 G342 V343 E345 G375 D...,G213 A214 K219 L311 P336 G338 V339 E341 G371 D...,R39 K219 G376 G399,...,NaN,NaN,P07378,9000079.0,421,EKKSINECDLKGKKVLIRVDFNVPVKNGKITNDYRIRSALPTLKKV...,ADP,Nc1ncnc2n(cnc12)[CH]3O[CH](CO[P](O)(=O)O[P](O)...,adenosine-5'-diphosphate,exact
6,155c,A,2.500,BS01,HEM,A,1,C15 C18 H19 I46 A47 Y54 G55 I58 W70 Y78 V79 K9...,C15 C18 H19 I46 A47 Y54 G55 I58 W70 Y78 V79 K9...,NaN,...,NaN,NaN,P00096,180013.0,135,NEGDAAKGEKEFNKCKACHMIQAPDGTDIKGGKTGPNLYGVVGRKI...,HEM,CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=C...,protoporphyrin ix containing fe,exact
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82210,9exw,A,2.430,BS01,A1H7X,A,1,V230 Y233 W236 F266 G268 A270 E272 Q321,V14 Y17 W20 F50 G52 A54 E56 Q105,NaN,...,NaN,NaN,O96028,38748070.0,403,LLKYNVGDLVWSKVSGYPWWPCMVSADPLLHSYTKLKGQKKSARQY...,A1H7X,Cn1cnc(c2cc(CNc3cccnc3)ccc2C)c1c4ccc5NC(=O)COc5c4,7-[3-methyl-5-[2-methyl-5-[(pyridin-3-ylamino)...,exact
82211,9exy,B,1.699,BS01,A1H7Z,B,1,V230 Y233 W236 F266 F267 G268 A270 E272 Q321,V17 Y20 W23 F53 F54 G55 A57 E59 Q91,NaN,...,NaN,NaN,O96028,38748070.0,401,KDHLLKYNVGDLVWSKVSGYPWWPCMVSADPLLHSYTKLKGQAASA...,A1H7Z,Cc1onc(c2cc(CN3CCCCC3)ccc2C)c1c4ccc5NC(=O)COc5c4,7-[5-methyl-3-[2-methyl-5-(piperidin-1-ylmethy...,exact
82359,9ezj,C,2.612,BS01,ZIQ,C,1,R103 E105 W208 R211 T212 P213 F304,R64 E66 W148 R151 T152 P153 F244,NaN,...,NaN,NaN,P48775,39537789.0,501,LIYGNYLHLEKVLNAQELQSETKGNKIHDEHLFIITHQAYELWFKQ...,ZIQ,C[C](N)(Cc1c[nH]c2ccccc12)C(O)=O,alpha-methyl-l-tryptophan,exact
82360,9ezj,C,2.612,BS02,A1H8G,C,1,F72 H76 F129 F140 F153 Q154 S155 H328,F33 H37 F90 F101 F111 Q112 S113 H268,NaN,...,NaN,NaN,P48775,39537789.0,502,LIYGNYLHLEKVLNAQELQSETKGNKIHDEHLFIITHQAYELWFKQ...,A1H8G,CCOC(=O)C1=C(OC)N=C2[CH](CCCN2C1=O)N([CH](C)c3...,ethyl (9~{r})-2-methoxy-4-oxidanylidene-9-[[(1...,exact


In [7]:
neg_df = add_negatives(
    biol_filtered,
    seq_col="protein_sequence",
    lig_col="smiles",
    neg_per_pos=1,
    seed=42,
    max_tries=1000,
)

Initial Dataset Shape: (36318, 26)
Total Unique Ligands Available: 5049
Total Unique Proteins to Process: 19917
Generated 24754 negative samples.
Final Output Label Distribution:
label
1    36318
0    24754
Name: count, dtype: int64


In [8]:
neg_df.to_csv(
    _P.data_processed_datasets / "biolip_cleaned_with_negatives.csv",
    index=False
)

# Dataprep for DDG and ThermoStab

In [9]:
fireprot = pd.read_csv(_P.data_raw_fireprotdb_dir / "fireprotdb_ec.tsv", sep="\t")

# Coerce to numeric to drop non-numeric placeholders if any
fireprot["DDG"] = pd.to_numeric(fireprot["DDG"], errors="coerce")
fireprot["DTM"] = pd.to_numeric(fireprot["DTM"], errors="coerce")

ddg_df = fireprot.dropna(subset=["DDG"]).copy()
print(ddg_df.shape)
dtm_df = fireprot.dropna(subset=["DTM"]).copy()
print(dtm_df.shape)

/tmp/ipykernel_54579/3764318755.py:1: DtypeWarning: Columns (0: CM, 1: DG, 2: DTM, 3: REVERSIBILITY, 4: TM, 5: BUFFER, 6: BUFFER_CONC, 7: ION, 8: ION_CONC, 9: MEASURE, 10: METHOD, 11: ACTIVE_SITE, 12: BINDING_SITE, 13: CONSERVATION, 14: WWPDB, 15: B_FACTOR, 16: IN_TUNNEL, 17: IN_POCKET, 18: REFERENCING_DATASET) have mixed types. Specify dtype option on import or set low_memory=False.
  fireprot = pd.read_csv(_P.data_raw_fireprotdb_dir / "fireprotdb_ec.tsv", sep="\t")


(3992, 53)
(3514, 53)


In [10]:
ddg_df.to_csv(_P.data_interim_dir / "fireprotdb" / "fireprotdb_ddg.csv", index=False)
dtm_df.to_csv(_P.data_interim_dir / "fireprotdb" / "fireprotdb_dtm.csv", index=False)

# Dataprep for phOpt (BRENDA)

In [ ]:
import json

with open("../data/raw/brenda/brenda_2025_1.json", "r") as f:
    data = json.load(f)

def search_for_uniprot(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if "uni" in k.lower() or "access" in k.lower():
                print("FOUND KEY:", k)
            search_for_uniprot(v)
    elif isinstance(obj, list):
        for item in obj:
            search_for_uniprot(item)

search_for_uniprot(data)

def find_first_accession(obj, path="root"):
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path} -> {k}"
            if k == "accessions":
                print("FOUND at:", new_path)
                print("VALUE:", v)
                return True
            if find_first_accession(v, new_path):
                return True
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            if find_first_accession(item, f"{path}[{i}]"):
                return True
    return False

find_first_accession(data)

In [ ]:
brenda_data = data["data"]

ec_with_ph = next(
    ec for ec, ec_data in brenda_data.items()
    if isinstance(ec_data, dict) and "ph_optimum" in ec_data
)

ec_data = brenda_data[ec_with_ph]

print("ec:", ec_with_ph)
print("keys:", ec_data.keys())
print("ph_optimum[0]:", ec_data["ph_optimum"][0])
protein_id = ec_data["ph_optimum"][0]["proteins"][0]

print(ec_data["protein"][protein_id])
list(ec_data["protein"].values())[0]
list(data.keys())[:5]
brenda_data = data["data"]
ec_example = next(iter(brenda_data.values()))
print(ec_example.keys())
ec_with_ph = next(
    ec for ec, ec_data in brenda_data.items()
    if isinstance(ec_data, dict) and "ph_optimum" in ec_data
)
ec_data = brenda_data[ec_with_ph]
ec_data.keys()
ec = ec_with_ph  # o cualquier EC que estés viendo
ec_data = brenda_data[ec]

for pid, pdata in ec_data["protein"].items():
    print(pid, pdata.keys())
    break

ec: 1.1.1.1
keys: dict_keys(['id', 'protein', 'recommended_name', 'systematic_name', 'synonyms', 'reaction', 'reaction_type', 'source_tissue', 'localization', 'natural_substrates_products', 'substrates_products', 'turnover_number', 'km_value', 'ph_optimum', 'ph_range', 'specific_activity', 'temperature_optimum', 'temperature_range', 'cofactor', 'activating_compound', 'inhibitor', 'metals_ions', 'molecular_weight', 'posttranslational_modification', 'subunits', 'pi_value', 'application', 'protein_variants', 'cloned', 'crystallization', 'purification', 'renatured', 'general_stability', 'organic_solvent_stability', 'oxidation_stability', 'ph_stability', 'storage_stability', 'temperature_stability', 'reference', 'ki_value', 'ic50_value', 'kcat_km_value', 'expression', 'general_information'])
ph_optimum[0]: {'proteins': ['10'], 'references': ['49'], 'comment': 'oxidation of ethanol, isoenzyme 2, 3 and 4', 'value': '10.7'}
{'id': '10', 'organism': 'Rattus norvegicus', 'references': ['10', '49

In [ ]:
results = []

for ec, ec_data in brenda_data.items():
    if not isinstance(ec_data, dict) or "ph_optimum" not in ec_data:
        continue

    proteins = ec_data.get("protein", {})

    for entry in ec_data["ph_optimum"]:
        ph = entry.get("value")
        protein_ids = entry.get("proteins", [])

        for pid in protein_ids:
            p = proteins.get(str(pid), {})  # pid puede venir como '10'
            org = p.get("organism")

            accs = p.get("accessions", [])
            uniprot = accs[0] if accs else None  # si hay varios, tomas el primero

            results.append({
                "ec": ec,
                "protein_id": str(pid),
                "organism": org,
                "phopt": ph,
                "uniprot_from_brenda": uniprot,
            })

df_raw = pd.DataFrame(results)
total = len(df_raw)

numeric = pd.to_numeric(df_raw["phopt"], errors="coerce").notna().sum()

print("Total entries:", total)
print("Numeric entries:", numeric)
print("Non-numeric entries:", total - numeric)
print("Fraction non-numeric:", (total - numeric) / total)
non_numeric = df_raw[pd.to_numeric(df_raw["phopt"], errors="coerce").isna()]["phopt"].unique()
print("Non-numeric values:", non_numeric[:20])

Total entries: 53256
Numeric entries: 44175
Non-numeric entries: 9081
Fraction non-numeric: 0.17051599819738622
Non-numeric values: <StringArray>
['7.7-8.6', '6.7-7.3',     '6-7',     '8-9',   '5.5-6',   '6.5-7', '8.6-8.8',
   '8-8.5', '8.5-9.1', '6.9-7.5',    '9-10',   '8-8.4', '5.6-6.5', '8.5-8.8',
 '10-10.5',   '7-7.5',   '6.7-7',   '10-11', '5.7-6.2', '6.5-7.5']
Length: 20, dtype: str


In [ ]:
import re

def parse_ph(x):
    if pd.isna(x):
        return None
    
    x = str(x)
    
    # case range tipe a-b
    if "-" in x:
        parts = x.split("-")
        try:
            low = float(parts[0])
            high = float(parts[1])
            return (low + high) / 2
        except:
            return None
    
    # case simple
    try:
        return float(x)
    except:
        return None


df_raw["phopt"] = df_raw["phopt"].apply(parse_ph)
print(df_raw.shape)

(53256, 5)


In [ ]:
df_raw["phopt"] = df_raw["phopt"].apply(parse_ph)
df_raw = df_raw.dropna(subset=["phopt"])

df_one = (
    df_raw.groupby(["ec", "protein_id", "organism", "uniprot_from_brenda"], as_index=False)
      .agg(
          phopt_median=("phopt", "median"),
          n_measurements=("phopt", "count"),
          phopt_std=("phopt", "std")
      )
)
df_one.to_csv(_P.data_interim_dir / 'phopt' / 'brenda_phopt.csv', index=False)